## Bladder dataset conversion to classifier format
Converts bladder_cell_data_part_*.json to *_cls.json with numeric labels.

In [ ]:
import json
from pathlib import Path
from collections import Counter

# ============================================================
# Configuration
# ============================================================
GENES_PREFIX = "Genes: "

# Only take original inputs; do NOT reprocess already-sorted outputs
src_files = sorted(Path("data").glob("heart_cell_data_part_*.json"))
src_files = [p for p in src_files if "_sorted" not in p.stem and "_cls" not in p.stem]

print(f"found {len(src_files)} source files (filtered)")

# Output folder
out_dir = Path("sorted_data")
out_dir.mkdir(parents=True, exist_ok=True)
print(f"writing sorted outputs to: {out_dir.resolve()}")

# ============================================================
# Robust gene parser
# ============================================================
def parse_genes(input_str):
    """
    Safely parse:
    'Genes: gene1 count1 gene2 count2 ...'
    Returns list[(gene, count_str)]
    """
    if not isinstance(input_str, str):
        return []
    if not input_str.startswith(GENES_PREFIX):
        return []

    tokens = input_str[len(GENES_PREFIX):].split()
    pairs = []
    for i in range(0, len(tokens) - 1, 2):  # safe for odd token counts
        gene = tokens[i]
        count = tokens[i + 1]
        pairs.append((gene, count))
    return pairs

# ============================================================
# Step 1: Compute global gene frequency
# (frequency = number of times gene appears across all rows)
# ============================================================
gene_freq = Counter()
odd_token_rows = 0
total_rows = 0
bad_prefix_rows = 0

for src in src_files:
    with src.open() as f:
        data = json.load(f)

    for row in data:
        total_rows += 1
        input_str = row.get("input", "")

        if not (isinstance(input_str, str) and input_str.startswith(GENES_PREFIX)):
            bad_prefix_rows += 1
            continue

        tokens = input_str[len(GENES_PREFIX):].split()
        if len(tokens) % 2 != 0:
            odd_token_rows += 1

        genes = parse_genes(input_str)
        gene_freq.update(g for g, _ in genes)

print(f"unique genes: {len(gene_freq)}")
print(f"rows with odd gene token count: {odd_token_rows} / {total_rows}")
print(f"rows missing/invalid '{GENES_PREFIX}' prefix: {bad_prefix_rows} / {total_rows}")

# ============================================================
# Step 2: Rewrite datasets with frequency-sorted genes
# Output: sorted_data/<original_stem>_sorted.json
# ============================================================
rows_with_no_genes = 0

for src in src_files:
    with src.open() as f:
        data = json.load(f)

    sorted_rows = []

    for row in data:
        input_str = row.get("input", "")
        genes = parse_genes(input_str)

        if not genes:
            rows_with_no_genes += 1
            # Keep row unchanged if genes can't be parsed
            sorted_rows.append(row)
            continue

        # Stable sort: by global frequency desc, then original position
        indexed = list(enumerate(genes))
        indexed.sort(key=lambda x: (-gene_freq[x[1][0]], x[0]))

        reordered = [genes[i] for i, _ in indexed]

        new_input = GENES_PREFIX + " ".join(f"{g} {c}" for g, c in reordered)

        new_row = dict(row)   # preserve all fields: instruction/output/label/etc.
        new_row["input"] = new_input
        sorted_rows.append(new_row)

    dest = out_dir / f"{src.stem}_sorted.json"
    with dest.open("w") as f:
        json.dump(sorted_rows, f, ensure_ascii=True, indent=2)

    print(f"wrote {dest} ({len(sorted_rows)} rows)")

print(f"rows kept unchanged due to unparsable/empty genes: {rows_with_no_genes}")


found 11 source files (filtered)
writing sorted outputs to: C:\Users\Hang Yu\Desktop\SCAP\Single_Cell_Aging_Prediction_LLaMA-Factory\sorted_data
unique genes: 4492
rows with odd gene token count: 2162 / 3859
rows missing/invalid 'Genes: ' prefix: 0 / 3859
wrote sorted_data\liver_cell_data_part_1_sorted.json (351 rows)
wrote sorted_data\liver_cell_data_part_10_sorted.json (350 rows)
wrote sorted_data\liver_cell_data_part_11_sorted.json (350 rows)
wrote sorted_data\liver_cell_data_part_2_sorted.json (351 rows)
wrote sorted_data\liver_cell_data_part_3_sorted.json (351 rows)
wrote sorted_data\liver_cell_data_part_4_sorted.json (351 rows)
wrote sorted_data\liver_cell_data_part_5_sorted.json (351 rows)
wrote sorted_data\liver_cell_data_part_6_sorted.json (351 rows)
wrote sorted_data\liver_cell_data_part_7_sorted.json (351 rows)
wrote sorted_data\liver_cell_data_part_8_sorted.json (351 rows)
wrote sorted_data\liver_cell_data_part_9_sorted.json (351 rows)
rows kept unchanged due to unparsable/